# 多 EDF 文件批量读取与训练流程（带讲解版）

这个 notebook 用来处理后续正式采集到的 **30 多个 EDF 文件**。

整体流程：

```text
多个 EDF 文件
↓
逐个读取
↓
每个文件切 focus / unfocus
↓
每个文件预处理
↓
每个文件提 STFT 功率谱特征
↓
把所有文件的特征拼成一个大 X
↓
把所有标签拼成一个大 y
↓
按 EDF 文件分组划分训练集 / 测试集
↓
StandardScaler + PCA + SVM 训练
```

默认假设每个 EDF 是 **20 分钟连续记录**：

```text
前 10 分钟 = focus / 专注
后 10 分钟 = unfocus / 非专注
```

如果后面改成三分类、四分类，或者 Trigger 能正常打标，需要改“切分标签”那一部分。


## 0. 安装依赖

如果还没装过，先在终端运行：

```bash
pip install mne numpy pandas scipy scikit-learn matplotlib joblib
```


In [1]:
# =========================
# 0. 导入库
# =========================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy import signal

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline

import joblib

warnings.filterwarnings("ignore")


## 1. 参数设置

这是最常改的地方。

建议把正式采集的 30 多个 EDF 文件放进同一个文件夹，例如：

```text
newdata/
    formal_20min/
        subject_01.edf
        subject_02.edf
        subject_03.edf
        ...
```

然后把 `EDF_DIR` 改成这个文件夹路径。


In [2]:
# =========================
# 1. 参数设置
# =========================

# 存放多个正式 EDF 文件的文件夹
EDF_DIR = Path(r".\newdata\formal_20min")

# 原项目使用 128 Hz，所以统一重采样到 128 Hz
TARGET_FS = 128

# 每个 EDF 的实验设计：前 10 分钟 focus，后 10 分钟 unfocus
TOTAL_MINUTES = 20
FOCUS_MINUTES = 10
UNFOCUS_MINUTES = 10

# 多文件正式训练默认不重复短数据
REPEAT_SHORT_DEMO_TO_20MIN = False

# 第一版使用这些标准 EEG 通道
# 暂时排除 Trigger、CM、X1/X2/X3、A1/A2
useful_channel = [
    "P3-Pz", "C3-Pz", "F3-Pz", "Fz-Pz",
    "F4-Pz", "C4-Pz", "P4-Pz", "Cz-Pz",
    "Fp1-Pz", "Fp2-Pz",
    "T3-Pz", "T5-Pz", "O1-Pz", "O2-Pz",
    "F7-Pz", "F8-Pz", "T6-Pz", "T4-Pz",
]

# 预处理参数
LOWCUT = 0.2
HIGHCUT = 43.0
FILTER_ORDER = 5

# STFT 参数，尽量贴近原项目
STFT_NPERSEG = 128
STFT_NFFT = 1024
STFT_NOVERLAP = 0

# STFT 后只保留这个频段的功率谱特征
FEATURE_FREQ_LOW = 0.5
FEATURE_FREQ_HIGH = 43.0

# 训练参数
RANDOM_STATE = 42
TEST_SIZE = 0.2

# 模型保存路径
MODEL_OUT_PATH = Path("svm_pca_edf_model.joblib")


## 2. 自动收集所有 EDF 文件

这句：

```python
edf_files = sorted(EDF_DIR.glob("*.edf"))
```

意思是：在 `EDF_DIR` 文件夹里找所有 `.edf` 文件，按文件名排序，放进 `edf_files`。

后面就可以：

```python
for edf_file in edf_files:
    ...
```

批量处理所有文件。


In [5]:
# =========================
# 2. 自动收集 EDF 文件
# =========================

edf_files = sorted(EDF_DIR.glob("*.edf"))

print("EDF 文件夹:", EDF_DIR)
print("找到 EDF 文件数量:", len(edf_files))

for i, f in enumerate(edf_files):
    print(i, f.name)

if len(edf_files) == 0:
    print("\n没有找到 EDF 文件。请检查 EDF_DIR 路径是否正确，以及文件后缀是否为 .edf")


EDF 文件夹: newdata\formal_20min
找到 EDF 文件数量: 2
0 data_0001_raw copy.edf
1 data_0001_raw.edf


## 3. EDF 读取函数

这一步替代原项目的 `.mat` 读取部分。

原项目大概是：

```python
data = scipy.io.loadmat(filename)
data = data['o']['data'][0][0]
data = data[:20 * 128 * 60, 3:17]
```

现在改成：

```python
raw = mne.io.read_raw_edf(...)
eeg_raw = raw.copy().pick("eeg")
data = eeg_raw.get_data().T * 1e6
```

关键点：

- `Trigger` 是事件标记通道，不进入 EEG 特征。
- `MNE` 读出的 EEG 通常是 V，所以乘 `1e6` 转成 μV。
- `get_data()` 原本是 `(通道数, 采样点数)`，转置成 `(采样点数, 通道数)`。
- 每个 EDF 都重采样到 `128 Hz`。


In [6]:
# =========================
# 3. EDF 读取函数
# =========================

def simplify_channel_name(name):
    """
    清理通道名前缀。
    例如 'EEG F3-Pz' -> 'F3-Pz'。
    """
    name = name.strip()
    if name.upper().startswith("EEG "):
        name = name[4:].strip()
    return name


def load_edf_like_original(edf_path, useful_channel, target_fs=128, verbose=False):
    """
    读取单个 EDF 文件，并整理成原项目后续代码需要的格式。
    
    返回：
    data_uv: shape = (n_samples, n_channels)，单位约为 μV
    fs: 采样率
    channel_names: 实际使用的 EEG 通道名
    trigger_info: Trigger 的简单信息
    """
    raw = mne.io.read_raw_edf(
        str(edf_path),
        preload=True,
        infer_types=True,
        verbose=verbose
    )
    
    # 清理通道名
    rename_dict = {ch: simplify_channel_name(ch) for ch in raw.ch_names}
    raw.rename_channels(rename_dict)
    
    # 单独查看 Trigger，但不把它作为 EEG 特征
    trigger_info = {
        "has_trigger": False,
        "trigger_channels": [],
        "trigger_unique": None,
        "trigger_nonzero_count": 0,
    }
    
    try:
        stim_raw = raw.copy().pick("stim")
        if len(stim_raw.ch_names) > 0:
            trigger_data = stim_raw.get_data()[0]
            trigger_info["has_trigger"] = True
            trigger_info["trigger_channels"] = stim_raw.ch_names
            trigger_info["trigger_unique"] = np.unique(trigger_data)
            trigger_info["trigger_nonzero_count"] = int(np.count_nonzero(trigger_data))
    except Exception:
        pass
    
    # 只保留 EEG
    eeg_raw = raw.copy().pick("eeg")
    
    # 检查通道是否齐全
    missing = [ch for ch in useful_channel if ch not in eeg_raw.ch_names]
    if missing:
        raise ValueError(
            f"{Path(edf_path).name} 缺少以下通道：\n"
            f"{missing}\n\n"
            f"当前 EDF 的 EEG 通道为：\n"
            f"{eeg_raw.ch_names}"
        )
    
    # 按 useful_channel 的顺序取通道，保证所有文件通道顺序一致
    eeg_raw.pick(useful_channel)
    
    # 重采样到原项目采样率
    if int(round(eeg_raw.info["sfreq"])) != target_fs:
        eeg_raw.resample(target_fs, npad="auto")
    
    fs = int(round(eeg_raw.info["sfreq"]))
    
    # MNE EEG 数据通常是 V，转成 μV
    # shape: (channels, times) -> (times, channels)
    data_uv = eeg_raw.get_data().T * 1e6
    
    channel_names = eeg_raw.ch_names
    
    return data_uv, fs, channel_names, trigger_info


## 4. 单文件读取测试

正式批量处理前，先拿第一个 EDF 测试一下。

重点检查：

- `data.shape` 是否合理。
- `fs` 是否是 128。
- 通道是否是我们选的 `useful_channel`。
- Trigger 有没有非零值。
- 数据幅值是否明显离谱。


In [7]:
# =========================
# 4. 单文件读取测试
# =========================

if len(edf_files) > 0:
    test_file = edf_files[0]
    print("测试读取:", test_file.name)
    
    data_test, fs_test, channel_names_test, trigger_info_test = load_edf_like_original(
        test_file,
        useful_channel=useful_channel,
        target_fs=TARGET_FS,
        verbose=True
    )
    
    print("\n========== 单文件测试结果 ==========")
    print("data shape:", data_test.shape)
    print("fs:", fs_test)
    print("duration_s:", data_test.shape[0] / fs_test)
    print("channel_names:", channel_names_test)
    print("mean/std/min/max:", np.mean(data_test), np.std(data_test), np.min(data_test), np.max(data_test))
    print("trigger_info:", trigger_info_test)
else:
    print("没有 EDF 文件，跳过测试。")


测试读取: data_0001_raw copy.edf
Extracting EDF parameters from newdata\formal_20min\data_0001_raw copy.edf...
Channel 'EEG P3-Pz' recognized as type EEG (renamed to 'P3-Pz').
Channel 'EEG C3-Pz' recognized as type EEG (renamed to 'C3-Pz').
Channel 'EEG F3-Pz' recognized as type EEG (renamed to 'F3-Pz').
Channel 'EEG Fz-Pz' recognized as type EEG (renamed to 'Fz-Pz').
Channel 'EEG F4-Pz' recognized as type EEG (renamed to 'F4-Pz').
Channel 'EEG C4-Pz' recognized as type EEG (renamed to 'C4-Pz').
Channel 'EEG P4-Pz' recognized as type EEG (renamed to 'P4-Pz').
Channel 'EEG Cz-Pz' recognized as type EEG (renamed to 'Cz-Pz').
Channel 'EEG CM-Pz' recognized as type EEG (renamed to 'CM-Pz').
Channel 'EEG A1-Pz' recognized as type EEG (renamed to 'A1-Pz').
Channel 'EEG Fp1-Pz' recognized as type EEG (renamed to 'Fp1-Pz').
Channel 'EEG Fp2-Pz' recognized as type EEG (renamed to 'Fp2-Pz').
Channel 'EEG T3-Pz' recognized as type EEG (renamed to 'T3-Pz').
Channel 'EEG T5-Pz' recognized as type EEG (

## 5. 批量读取并切分 focus / unfocus

这里开始正式批量处理。

使用两个字典保存原始切分后的数据：

```python
focus_raw[recordname]
unfocus_raw[recordname]
```

例如：

```python
focus_raw["subject_01"]
unfocus_raw["subject_01"]
```

每个 EDF 都被切成：

```text
前 10 分钟 → focus_raw
后 10 分钟 → unfocus_raw
```

如果某个 EDF 不够 20 分钟，会被跳过。


In [8]:
# =========================
# 5. 批量读取并切分 focus / unfocus
# =========================

focus_raw = {}
unfocus_raw = {}

file_info_list = []

all_channel_names = None
all_fs = None

required_n = TOTAL_MINUTES * 60 * TARGET_FS
focus_n = FOCUS_MINUTES * 60 * TARGET_FS

print("每个 EDF 需要采样点数:", required_n)
print("focus 采样点数:", focus_n)
print("unfocus 采样点数:", required_n - focus_n)

for edf_file in edf_files:
    recordname = edf_file.stem
    
    print("\n==============================")
    print("正在读取:", recordname)
    print("==============================")
    
    try:
        data, fs, channel_names, trigger_info = load_edf_like_original(
            edf_file,
            useful_channel=useful_channel,
            target_fs=TARGET_FS,
            verbose=False
        )
    except Exception as e:
        print("读取失败，跳过:", recordname)
        print("错误信息:", e)
        continue
    
    duration_s = data.shape[0] / fs
    
    # 正式数据必须够 20 分钟
    if data.shape[0] < required_n:
        print(f"警告：{recordname} 数据长度不足，跳过。")
        print("当前采样点:", data.shape[0], "需要采样点:", required_n)
        print("当前时长秒:", duration_s)
        continue
    
    # 检查通道顺序是否一致
    if all_channel_names is None:
        all_channel_names = channel_names
    else:
        if channel_names != all_channel_names:
            raise ValueError(
                f"{recordname} 的通道顺序和前面文件不一致！\n"
                f"当前: {channel_names}\n"
                f"标准: {all_channel_names}"
            )
    
    # 检查采样率是否一致
    if all_fs is None:
        all_fs = fs
    else:
        if fs != all_fs:
            raise ValueError(f"{recordname} 的采样率不一致：{fs} vs {all_fs}")
    
    # 只取前 20 分钟
    data = data[:required_n, :]
    
    # 前 10 分钟 focus，后 10 分钟 unfocus
    focus_raw[recordname] = data[:focus_n, :]
    unfocus_raw[recordname] = data[focus_n:required_n, :]
    
    file_info_list.append({
        "recordname": recordname,
        "file": str(edf_file),
        "duration_s_after_resample": duration_s,
        "n_samples_used": required_n,
        "fs": fs,
        "n_channels": len(channel_names),
        "has_trigger": trigger_info["has_trigger"],
        "trigger_nonzero_count": trigger_info["trigger_nonzero_count"],
        "mean_uv": float(np.mean(data)),
        "std_uv": float(np.std(data)),
        "min_uv": float(np.min(data)),
        "max_uv": float(np.max(data)),
    })
    
    print("data shape:", data.shape)
    print("focus shape:", focus_raw[recordname].shape)
    print("unfocus shape:", unfocus_raw[recordname].shape)
    print("trigger 非零数量:", trigger_info["trigger_nonzero_count"])

print("\n最终有效 EDF 文件数量:", len(focus_raw))


每个 EDF 需要采样点数: 153600
focus 采样点数: 76800
unfocus 采样点数: 76800

正在读取: data_0001_raw copy
警告：data_0001_raw copy 数据长度不足，跳过。
当前采样点: 320 需要采样点: 153600
当前时长秒: 2.5

正在读取: data_0001_raw
警告：data_0001_raw 数据长度不足，跳过。
当前采样点: 320 需要采样点: 153600
当前时长秒: 2.5

最终有效 EDF 文件数量: 0


## 6. 查看批量读取汇总表

这张表用来快速发现异常文件：

- 某些文件时长不足。
- 某些文件幅值特别离谱。
- 某些文件 Trigger 有非零值。
- 某些文件标准差特别大，可能电极接触有问题。


In [9]:
# =========================
# 6. 文件读取汇总表
# =========================

file_info_df = pd.DataFrame(file_info_list)

if len(file_info_df) > 0:
    display(file_info_df)
else:
    print("没有成功读取的 EDF 文件。")


没有成功读取的 EDF 文件。


## 7. 画一个文件的原始信号检查

随便选一个文件、一个通道，画前 10 秒看看。

这一步只是肉眼检查：

- 有没有明显全 0。
- 有没有离谱尖峰。
- 幅值是否大致合理。


In [10]:
# =========================
# 7. 原始信号可视化
# =========================

if len(focus_raw) > 0:
    example_record = list(focus_raw.keys())[0]
    channel_to_plot = "F7-Pz"
    channel_idx = all_channel_names.index(channel_to_plot)
    
    plot_seconds = 10
    plot_n = plot_seconds * TARGET_FS
    
    plt.figure(figsize=(12, 4))
    plt.plot(focus_raw[example_record][:plot_n, channel_idx])
    plt.title(f"Raw Focus Signal: {example_record}, {channel_to_plot}, first {plot_seconds}s")
    plt.xlabel("Samples")
    plt.ylabel("Amplitude (μV)")
    plt.grid(True)
    plt.show()
else:
    print("没有可画的数据。")


没有可画的数据。


## 8. 预处理函数

自采 EDF 数据已经是 μV 物理单位，不像原复现数据那样有 4000 左右的大偏置。

所以这里采用：

```text
每个通道减均值
↓
0.2–43 Hz Butterworth 带通滤波
```

这相当于保留原项目“去 baseline + 带通滤波”的思想，但更适合自采 EDF。

注意：不要重复做两次同样的 0.2–43 Hz 带通滤波。


In [11]:
# =========================
# 8. 预处理函数
# =========================

def butter_bandpass_filter(data_1d, lowcut, highcut, fs, order=5):
    """
    对单通道一维 EEG 数据做 Butterworth 带通滤波。
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    
    if not (0 < low < high < 1):
        raise ValueError(
            f"滤波频率设置错误：lowcut={lowcut}, highcut={highcut}, fs={fs}"
        )
    
    b, a = signal.butter(order, [low, high], btype="band")
    filtered = signal.filtfilt(b, a, data_1d)
    return filtered


def preprocess_like_original(raw_data, fs):
    """
    raw_data shape = (n_samples, n_channels)
    
    返回：
    filtered_data shape = (n_samples, n_channels)
    """
    # 每个通道去均值
    detrend_data = raw_data - np.mean(raw_data, axis=0, keepdims=True)
    
    # 每个通道分别滤波
    filtered_data = np.zeros_like(detrend_data, dtype=float)
    
    for col in range(detrend_data.shape[1]):
        filtered_data[:, col] = butter_bandpass_filter(
            detrend_data[:, col],
            lowcut=LOWCUT,
            highcut=HIGHCUT,
            fs=fs,
            order=FILTER_ORDER
        )
    
    return filtered_data


## 9. 批量预处理

对每一个 EDF 的 focus 和 unfocus 分别预处理。

结果保存到：

```python
focus_filtered[recordname]
unfocus_filtered[recordname]
```


In [12]:
# =========================
# 9. 批量预处理
# =========================

focus_filtered = {}
unfocus_filtered = {}

for i, recordname in enumerate(focus_raw.keys()):
    print(f"[{i+1}/{len(focus_raw)}] 正在预处理:", recordname)
    
    focus_filtered[recordname] = preprocess_like_original(
        focus_raw[recordname],
        fs=TARGET_FS
    )
    
    unfocus_filtered[recordname] = preprocess_like_original(
        unfocus_raw[recordname],
        fs=TARGET_FS
    )

print("\n预处理完成。")
print("focus_filtered 文件数:", len(focus_filtered))
print("unfocus_filtered 文件数:", len(unfocus_filtered))



预处理完成。
focus_filtered 文件数: 0
unfocus_filtered 文件数: 0


## 10. 预处理前后对比

检查滤波后是否围绕 0 附近波动。

正式 20 分钟连续数据不会像之前 2.5 秒重复数据那样每隔 2.5 秒出现拼接伪影。


In [13]:
# =========================
# 10. 预处理前后对比
# =========================

if len(focus_filtered) > 0:
    example_record = list(focus_filtered.keys())[0]
    channel_to_plot = "F7-Pz"
    channel_idx = all_channel_names.index(channel_to_plot)
    
    plot_seconds = 10
    plot_n = plot_seconds * TARGET_FS
    
    plt.figure(figsize=(12, 8))
    
    plt.subplot(2, 1, 1)
    plt.plot(focus_raw[example_record][:plot_n, channel_idx])
    plt.title(f"Before Preprocess: {example_record}, {channel_to_plot}")
    plt.ylabel("Amplitude (μV)")
    plt.grid(True)
    
    plt.subplot(2, 1, 2)
    plt.plot(focus_filtered[example_record][:plot_n, channel_idx])
    plt.title(f"After Mean Removal + Bandpass: {example_record}, {channel_to_plot}")
    plt.xlabel("Samples")
    plt.ylabel("Amplitude")
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("没有可画的数据。")


没有可画的数据。


## 11. STFT 功率谱特征提取

这一步贴近原项目：

```python
signal.stft(
    data,
    fs=128,
    window=signal.windows.blackman(128),
    nperseg=128,
    nfft=1024,
    noverlap=0
)
```

注意这里用的是 **功率谱**：

```python
power = np.abs(Zxx) ** 2
```

而不是单纯的幅值：

```python
amplitude = np.abs(Zxx)
```

每个通道提取 STFT 功率谱后，把所有通道的特征横向拼接。


In [14]:
# =========================
# 11. STFT 功率谱特征提取
# =========================

def extract_stft_power_features(data_filtered, fs):
    """
    输入：
    data_filtered shape = (n_samples, n_channels)
    
    输出：
    X shape = (n_windows, n_features)
    
    特征：
    STFT 功率谱 power = |Zxx|^2
    """
    all_channel_features = []
    
    for col in range(data_filtered.shape[1]):
        f, t, Zxx = signal.stft(
            data_filtered[:, col],
            fs=fs,
            window=signal.windows.blackman(STFT_NPERSEG),
            nperseg=STFT_NPERSEG,
            nfft=STFT_NFFT,
            noverlap=STFT_NOVERLAP,
            boundary=None,
            padded=False
        )
        
        # 只保留目标频段
        freq_mask = (f >= FEATURE_FREQ_LOW) & (f <= FEATURE_FREQ_HIGH)
        
        # 功率谱：幅值平方
        power = np.abs(Zxx[freq_mask, :]) ** 2
        
        # power shape = (频率点数, 时间窗数)
        # 转成 (时间窗数, 频率点数)
        channel_feature = power.T
        
        all_channel_features.append(channel_feature)
    
    # 把不同通道的频率特征横向拼起来
    X = np.concatenate(all_channel_features, axis=1)
    
    return X


## 12. 构造总训练集 X / y / groups

这是多文件训练最核心的一步。

对每个 EDF：

```text
focus_filtered[recordname]   → X_focus, 标签 1
unfocus_filtered[recordname] → X_unfocus, 标签 0
```

然后：

```python
X_list.append(X_focus)
X_list.append(X_unfocus)
```

`append` 只是把每个文件的小矩阵先收集到列表里。

最后：

```python
X = np.concatenate(X_list, axis=0)
```

才是真正把所有小矩阵上下拼成一个大训练矩阵。


`groups` 是为了记录每个样本来自哪个 EDF 文件。

例如：

```text
subject_01 的所有窗口 → group 0
subject_02 的所有窗口 → group 1
subject_03 的所有窗口 → group 2
```

后面用 `GroupShuffleSplit` 可以保证：

```text
同一个 EDF 文件的所有时间窗，要么全部进训练集，要么全部进测试集
```

这样比普通随机划分更严格，不容易虚高。


In [15]:
# =========================
# 12. 构造总训练集 X / y / groups
# =========================

X_list = []
y_list = []
groups_list = []
recordname_list = []

for file_idx, recordname in enumerate(focus_filtered.keys()):
    print(f"[{file_idx+1}/{len(focus_filtered)}] 正在提取特征:", recordname)
    
    # focus: 标签 1
    X_focus = extract_stft_power_features(focus_filtered[recordname], fs=TARGET_FS)
    y_focus = np.ones(X_focus.shape[0], dtype=int)
    
    # unfocus: 标签 0
    X_unfocus = extract_stft_power_features(unfocus_filtered[recordname], fs=TARGET_FS)
    y_unfocus = np.zeros(X_unfocus.shape[0], dtype=int)
    
    # 先 append 到列表里
    X_list.append(X_focus)
    X_list.append(X_unfocus)
    
    y_list.append(y_focus)
    y_list.append(y_unfocus)
    
    # groups 记录每个样本来自哪个 EDF 文件
    groups_list.append(np.full(X_focus.shape[0], file_idx))
    groups_list.append(np.full(X_unfocus.shape[0], file_idx))
    
    recordname_list.append(recordname)
    
    print("  X_focus:", X_focus.shape, "X_unfocus:", X_unfocus.shape)

# 最后一次性拼成总 X/y/groups
X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)
groups = np.concatenate(groups_list, axis=0)

print("\n========== 总数据集 ==========")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("groups shape:", groups.shape)
print("标签分布:", dict(zip(*np.unique(y, return_counts=True))))
print("文件 group 数量:", len(np.unique(groups)))


ValueError: need at least one array to concatenate

## 13. 理解 X / y / groups 的含义

最终模型看到的是：

```python
X.shape = (总样本数, 每个样本的特征数)
y.shape = (总样本数,)
groups.shape = (总样本数,)
```

其中：

- `X[i]`：第 i 个时间窗的 STFT 功率谱特征。
- `y[i]`：第 i 个时间窗的标签，`1=focus`，`0=unfocus`。
- `groups[i]`：第 i 个时间窗来自第几个 EDF 文件。


In [16]:
# =========================
# 13. 查看前几个样本的标签和 group
# =========================

preview_n = min(20, len(y))

preview_df = pd.DataFrame({
    "sample_index": np.arange(preview_n),
    "label_y": y[:preview_n],
    "label_name": ["focus" if v == 1 else "unfocus" for v in y[:preview_n]],
    "group_file_idx": groups[:preview_n],
})

display(preview_df)


NameError: name 'y' is not defined

## 14. 按 EDF 文件分组划分训练集 / 测试集

普通 `train_test_split` 会随机打散所有时间窗，这可能造成数据泄漏：

```text
同一个 EDF 的一些时间窗进训练集
同一个 EDF 的另一些时间窗进测试集
```

因为同一个 EDF 里的相邻时间窗太像了，这样测试集会“太熟悉”，准确率可能虚高。

所以这里优先用 `GroupShuffleSplit`：

```python
train_idx, test_idx = next(gss.split(X, y, groups=groups))
```

其中 `next(...)` 是从划分生成器里取出第一组训练/测试下标。

`n_splits=1` 表示只生成一组划分。


In [ ]:
# =========================
# 14. 按文件分组划分训练集 / 测试集
# =========================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

groups_train = groups[train_idx]
groups_test = groups[test_idx]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\n训练集标签分布:", dict(zip(*np.unique(y_train, return_counts=True))))
print("测试集标签分布:", dict(zip(*np.unique(y_test, return_counts=True))))

print("\n训练集 EDF group:", sorted(np.unique(groups_train).tolist()))
print("测试集 EDF group:", sorted(np.unique(groups_test).tolist()))


## 15. 训练模型：StandardScaler + PCA + SVM

这里做成 `Pipeline`，比手动一行行写更干净。

流程是：

```text
StandardScaler：标准化特征
↓
PCA：降维，保留 95% 方差信息
↓
SVC：RBF 核 SVM 分类
```

这个和原项目的 `PCA + SVM` 思路一致，只是把标准化也放进来了。

标准化对 SVM 和 PCA 都很重要，因为功率谱不同特征的数值范围可能差很多。


In [ ]:
# =========================
# 15. 训练模型
# =========================

model_SVM_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ("svm", SVC(kernel="rbf", random_state=RANDOM_STATE, probability=True)),
])

model_SVM_pca.fit(X_train, y_train)

y_pred = model_SVM_pca.predict(X_test)

print("训练完成。")
print("PCA 后维度:", model_SVM_pca.named_steps["pca"].n_components_)


## 16. 查看分类结果

`classification_report` 里：

- `precision`：模型预测成某一类时，有多少是真的。
- `recall`：真实属于某一类时，有多少被找出来。
- `f1-score`：precision 和 recall 的综合。
- `support`：测试集中该类真实样本数量。
- `accuracy`：总体准确率。

注意：如果 EDF 文件数量少，按文件分组划分后测试集文件数量少，结果会波动很大。文件越多，评估越稳定。


In [ ]:
# =========================
# 16. 分类结果
# =========================

acc = accuracy_score(y_test, y_pred)

print("Accuracy:", acc)

print("\n混淆矩阵:")
print(confusion_matrix(y_test, y_pred))

print("\n分类报告:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["unfocus", "focus"]
))


## 17. 可选：普通随机划分对照

这一步不是正式推荐方案，只是用来对比。

普通随机划分可能准确率更高，但它可能是因为同一个 EDF 的相近时间窗同时出现在训练集和测试集里，存在“数据泄漏”。

如果：

```text
随机划分准确率很高
按文件分组划分准确率明显下降
```

说明模型可能更多记住了某个记录的特征，而不是学到了稳定的 focus/unfocus 差异。


In [ ]:
# =========================
# 17. 可选：普通随机划分对照
# =========================

RUN_RANDOM_SPLIT_BASELINE = False

if RUN_RANDOM_SPLIT_BASELINE:
    X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )
    
    model_random = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
        ("svm", SVC(kernel="rbf", random_state=RANDOM_STATE, probability=True)),
    ])
    
    model_random.fit(X_train_r, y_train_r)
    y_pred_r = model_random.predict(X_test_r)
    
    print("随机划分 Accuracy:", accuracy_score(y_test_r, y_pred_r))
    print("\n随机划分分类报告:")
    print(classification_report(
        y_test_r,
        y_pred_r,
        target_names=["unfocus", "focus"]
    ))
else:
    print("当前未运行普通随机划分对照。需要时把 RUN_RANDOM_SPLIT_BASELINE 改成 True。")


## 18. 保存模型

保存的是整个 pipeline：

```text
StandardScaler + PCA + SVM
```

以后预测新数据时，也要使用同样的预处理和 STFT 特征提取，然后调用这个模型。


In [ ]:
# =========================
# 18. 保存模型
# =========================

joblib.dump(model_SVM_pca, MODEL_OUT_PATH)

print("模型已保存到:", MODEL_OUT_PATH)


## 19. 之后最可能要改的地方

### 情况 A：通道名变了

如果正式 EDF 通道名不再是 `F7-Pz`，而是 `F7` 或 `EEG F7-Pz`，优先检查读取出来的通道名，然后修改 `useful_channel` 或 `simplify_channel_name()`。

---

### 情况 B：实验不是前 10 分钟 / 后 10 分钟

现在切分逻辑是：

```python
focus_raw[recordname] = data[:focus_n, :]
unfocus_raw[recordname] = data[focus_n:required_n, :]
```

如果之后是三分类，例如：

```text
0-5 min focus
5-10 min distract
10-15 min fatigue
```

那就要新增：

```python
distract_raw
fatigue_raw
```

或者改成更通用的 `segments` 列表。

---

### 情况 C：Trigger 能正常打标

如果正式实验 Trigger 里有非零值，可以后续根据 Trigger 自动切段。那样比固定时间切更稳。

---

### 情况 D：想复现原项目更严格

如果原项目使用了 `BaselineRemoval.IModPoly()`，可以做一个对照版本：

```text
每通道去均值 + 带通
vs
IModPoly + 带通
```

比较两者效果。
